# 03 — Particionamiento y plan de Spark

Compara la lectura de las interacciones por año y dos configuraciones de shuffle. Es un experimento local descriptivo: los tiempos dependen del equipo, el caché del sistema y las ejecuciones concurrentes. No modifica los datos oficiales.


In [ ]:
import platform
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import display
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.processing.build_fact_award import configure_pyspark_python  # noqa: E402
from src.processing.build_fact_award_item import configure_windows_hadoop  # noqa: E402

data = ROOT / 'data' / 'processed' / 'entity_supplier_year'
if not (data / '_SUCCESS').is_file():
    raise FileNotFoundError(f'Falta la tabla publicada: {data}')
hadoop = configure_windows_hadoop()
configure_pyspark_python()
builder = (SparkSession.builder.appName('parte1-particionamiento')
           .master('local[4]').config('spark.sql.adaptive.enabled', 'false'))
if hadoop:
    builder = builder.config('spark.driver.extraLibraryPath', str(hadoop / 'bin'))
spark = builder.getOrCreate()
frame = spark.read.option('pathGlobFilter', '*.parquet').parquet(str(data))
print('Sistema:', platform.platform(), '| Python:', platform.python_version(),
      '| Spark:', spark.version)
print('Master:', spark.sparkContext.master, '| AQE:', spark.conf.get('spark.sql.adaptive.enabled'))
print('Archivos Parquet:', len(list(data.rglob('*.parquet'))))
print('Particiones por año:', sorted(p.name for p in data.glob('award_year=*')))
print('Plan sin filtro de año:')
frame.groupBy('entity_id').agg(F.sum('award_count').alias('total')).explain()
print('Plan con filtro 2023:')
frame.where(F.col('award_year') == 2023).groupBy('entity_id').agg(
    F.sum('award_count').alias('total')).explain()

results = []
for partitions in (4, 8):
    spark.conf.set('spark.sql.shuffle.partitions', str(partitions))
    for repetition in range(3):
        start = time.perf_counter()
        query = frame.where(F.col('award_year') == 2023).groupBy('entity_id').agg(
            F.sum('award_count').alias('total'))
        result = query.agg(F.count('*').alias('entities'), F.sum('total').alias('awards')).first()
        results.append({'shuffle': partitions, 'repeticion': repetition,
                        'tipo': 'calentamiento' if repetition == 0 else 'medición',
                        'segundos': round(time.perf_counter() - start, 3),
                        'entidades': result.entities, 'adjudicaciones': result.awards})
display(pd.DataFrame(results))
spark.stop()
